In [1]:
################################################################################
# BASIC IMPORTS
################################################################################
import os
import pandas as pd
import numpy as np
import librosa
import random
from tqdm import tqdm
from pathlib import Path
import torch
import torchaudio
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import tensorflow as tf
import tensorflow_hub as hub

/Users/junichikoganemaru/miniconda3/envs/py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Check if GPU training is available on Mac
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")

tensor([1.], device='mps:0')


In [ ]:
################################################################################
# CONFIGURATION
################################################################################

class Config:
    MODEL_NAME = "IterativePseudoLabel_Ensemble"
    ABRIDGED_RUN = False # Set to True for quick testing

    # Paths
    DATA_DIR = "../data/"
    AUDIO_DIR = os.path.join(DATA_DIR, "train_audio/")
    UNLABELED_DIR = os.path.join(DATA_DIR, "unlabeled_soundscapes/")
    CHECKPOINT_DIR = "checkpoints/"
    INITIAL_PSEUDO_LABEL_CSV = "initial_pseudo_labels.csv"

    # Audio Preprocessing & Spectrogram
    SAMPLE_RATE = 32000
    SAMPLE_LENGTH_SECONDS = 5
    N_SAMPLES = SAMPLE_RATE * SAMPLE_LENGTH_SECONDS
    N_FFT = 2048
    HOP_LENGTH = 512
    N_MELS = 128

    # Training
    BATCH_SIZE = 64
    LEARNING_RATE = 0.001
    NUM_EPOCHS_PER_ITERATION = 5 # Epochs for each self-training loop
    NUM_EPOCHS_FINAL = 5 # Epochs for the final ensemble training
    NUM_SPECIES = 182

    # Advanced training parameters 
    NUM_ITERATIONS = 2 # Number of times to re-label and retrain for iterative pseudo-labeling
    NUM_ENSEMBLE_MODELS = 4 # Number of models to train for the final ensemble
    SMOOTHING_KERNEL = [0.1, 0.2, 0.4, 0.2, 0.1] # Kernel for smoothing predictions over time

    # Pseudo-Labeling
    PSEUDO_LABEL_MODEL_URL = 'https://www.kaggle.com/models/google/bird-vocalization-classifier/TensorFlow2/bird-vocalization-classifier/8'
    PSEUDO_LABEL_SAMPLES_PER_SOUNDSCAPE = 4

In [4]:
Path(Config.CHECKPOINT_DIR).mkdir(exist_ok=True)

In [5]:
SPECIES = ['asbfly', 'ashdro1', 'ashpri1', 'ashwoo2', 'asikoe2', 'asiope1', 'aspfly1', 'aspswi1', 'barfly1', 'barswa', 'bcnher', 'bkcbul1', 'bkrfla1', 'bkskit1', 'bkwsti', 'bladro1', 'blaeag1', 'blakit1', 'blhori1', 'blnmon1', 'blrwar1', 'bncwoo3', 'brakit1', 'brasta1', 'brcful1', 'brfowl1', 'brnhao1', 'brnshr', 'brodro1', 'brwjac1', 'brwowl1', 'btbeat1', 'bwfshr1', 'categr', 'chbeat1', 'cohcuc1', 'comfla1', 'comgre', 'comior1', 'comkin1', 'commoo3', 'commyn', 'compea', 'comros', 'comsan', 'comtai1', 'copbar1', 'crbsun2', 'cregos1', 'crfbar1', 'crseag1', 'dafbab1', 'darter2', 'eaywag1', 'emedov2', 'eucdov', 'eurbla2', 'eurcoo', 'forwag1', 'gargan', 'gloibi', 'goflea1', 'graher1', 'grbeat1', 'grecou1', 'greegr', 'grefla1', 'grehor1', 'grejun2', 'grenig1', 'grewar3', 'grnsan', 'grnwar1', 'grtdro1', 'gryfra', 'grynig2', 'grywag', 'gybpri1', 'gyhcaf1', 'heswoo1', 'hoopoe', 'houcro1', 'houspa', 'inbrob1', 'indpit1', 'indrob1', 'indrol2', 'indtit1', 'ingori1', 'inpher1', 'insbab1', 'insowl1', 'integr', 'isbduc1', 'jerbus2', 'junbab2', 'junmyn1', 'junowl1', 'kenplo1', 'kerlau2', 'labcro1', 'laudov1', 'lblwar1', 'lesyel1', 'lewduc1', 'lirplo', 'litegr', 'litgre1', 'litspi1', 'litswi1', 'lobsun2', 'maghor2', 'malpar1', 'maltro1', 'malwoo1', 'marsan', 'mawthr1', 'moipig1', 'nilfly2', 'niwpig1', 'nutman', 'orihob2', 'oripip1', 'pabflo1', 'paisto1', 'piebus1', 'piekin1', 'placuc3', 'plaflo1', 'plapri1', 'plhpar1', 'pomgrp2', 'purher1', 'pursun3', 'pursun4', 'purswa3', 'putbab1', 'redspu1', 'rerswa1', 'revbul', 'rewbul', 'rewlap1', 'rocpig', 'rorpar', 'rossta2', 'rufbab3', 'ruftre2', 'rufwoo2', 'rutfly6', 'sbeowl1', 'scamin3', 'shikra1', 'smamin1', 'sohmyn1', 'spepic1', 'spodov', 'spoowl1', 'sqtbul1', 'stbkin1', 'sttwoo1', 'thbwar1', 'tibfly3', 'tilwar1', 'vefnut1', 'vehpar1', 'wbbfly1', 'wemhar1', 'whbbul2', 'whbsho3', 'whbtre1', 'whbwag1', 'whbwat1', 'whbwoo2', 'whcbar1', 'whiter2', 'whrmun', 'whtkin2', 'woosan', 'wynlau1', 'yebbab1', 'yebbul3', 'zitcis1']
SPECIES_TO_INDEX = {s: i for i, s in enumerate(SPECIES)}
INDEX_TO_SPECIES = {i: s for s, i in SPECIES_TO_INDEX.items()}

In [ ]:
################################################################################
# FUNCTION FOR INITIAL PSEUDO-LABELING WITH GOOGLE MODEL
################################################################################

def generate_initial_pseudo_labels():
    """
    Generates the first set of pseudo-labels using the Google model. Outputs to `initial_pseudo_labels.csv` with 182 species.
    """
    if os.path.exists(Config.INITIAL_PSEUDO_LABEL_CSV):
        print(f"Loading cached initial pseudo-labels from {Config.INITIAL_PSEUDO_LABEL_CSV}")
        return pd.read_csv(Config.INITIAL_PSEUDO_LABEL_CSV)

    print("Generating initial pseudo-labels with Google's model...")
    try:
        import tensorflow as tf
        import tensorflow_hub as hub

        google_model = hub.load(Config.PSEUDO_LABEL_MODEL_URL)
        google_labels_path = hub.resolve(Config.PSEUDO_LABEL_MODEL_URL) + "/assets/label.csv"
        google_labels_df = pd.read_csv(google_labels_path)
        google_ebird_to_index = {name: i for i, name in enumerate(google_labels_df['ebird2021'])}

        unlabeled_soundscapes = sorted(os.listdir(Config.UNLABELED_DIR))
        if Config.ABRIDGED_RUN:
            unlabeled_soundscapes = random.sample(unlabeled_soundscapes, 5)

        new_rows = []
        for path in tqdm(unlabeled_soundscapes, desc="Initial Labeling"):
            full_path = os.path.join(Config.UNLABELED_DIR, path)
            try:
                waveform, _ = librosa.load(full_path, sr=Config.SAMPLE_RATE, mono=True)
                duration = librosa.get_duration(y=waveform, sr=Config.SAMPLE_RATE)
                for _ in range(Config.PSEUDO_LABEL_SAMPLES_PER_SOUNDSCAPE):
                    if len(waveform) < Config.N_SAMPLES: continue
                    start = np.random.randint(0, len(waveform) - Config.N_SAMPLES)
                    clip = waveform[start : start + Config.N_SAMPLES]
                    
                    logits = google_model.infer_tf(clip[np.newaxis, :])['label'].numpy()[0]
                    all_probs = torch.sigmoid(torch.from_numpy(logits)).numpy()


                    our_species_probs = np.zeros(Config.NUM_SPECIES, dtype=np.float32)

                    for i, species_name in enumerate(SPECIES):
                        if species_name in google_ebird_to_index:
                            google_idx = google_ebird_to_index[species_name]
                            our_species_probs[i] = all_probs[google_idx]

                    
                    primary_label_idx = np.argmax(our_species_probs)
                    
                    new_rows.append({
                        'filepath': full_path, 'primary_label': INDEX_TO_SPECIES[primary_label_idx],
                        'duration': duration, 'is_pseudo': True,
                        'soft_label': ','.join(map(str, our_species_probs)), 'start_time': start / Config.SAMPLE_RATE
                    })
            except Exception as e:
                print(f"Error processing {path}: {e}")
                continue

        pseudo_df = pd.DataFrame(new_rows)
        pseudo_df.to_csv(Config.INITIAL_PSEUDO_LABEL_CSV, index=False)
        print(f"Saved {len(pseudo_df)} initial pseudo-labels to {Config.INITIAL_PSEUDO_LABEL_CSV}")
        return pseudo_df

    except ImportError:
        print("\n---\nERROR: `tensorflow` and `tensorflow_hub` are required.\n---")
        exit()

In [ ]:
################################################################################
# HELPER FUNCTIONS (Dataset, Model, Training)
################################################################################

class BirdCLEFDataset(Dataset):
    def __init__(self, df, is_train=True):
        self.df = df
        self.is_train = is_train
        self.mel_spectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=Config.SAMPLE_RATE, n_fft=Config.N_FFT, hop_length=Config.HOP_LENGTH, n_mels=Config.N_MELS)
        self.augmentations = nn.Sequential(
            torchaudio.transforms.FrequencyMasking(freq_mask_param=8),
            torchaudio.transforms.TimeMasking(time_mask_param=40))

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y, _ = librosa.load(row.filepath, sr=Config.SAMPLE_RATE, mono=True)
        
        if row.get('is_pseudo', False):
            start_sample = int(row.start_time * Config.SAMPLE_RATE)
            y = y[start_sample : start_sample + Config.N_SAMPLES]
        else:
            if len(y) > Config.N_SAMPLES:
                start = np.random.randint(0, len(y) - Config.N_SAMPLES)
                y = y[start : start + Config.N_SAMPLES]
            else:
                y = np.pad(y, (0, Config.N_SAMPLES - len(y)), 'constant')

        spec = self.mel_spectrogram(torch.from_numpy(y).float())
        spec = (spec - spec.mean()) / spec.std()
        if self.is_train: spec = self.augmentations(spec)
        spec = spec.unsqueeze(0).repeat(3, 1, 1)

        if row.get('is_pseudo', False):
            soft_label_parts = row.soft_label.split(',')
            label = torch.tensor([float(p) for p in soft_label_parts], dtype=torch.float)
        else:
            label_idx = SPECIES_TO_INDEX[row.primary_label]
            label = torch.nn.functional.one_hot(torch.tensor(label_idx), num_classes=Config.NUM_SPECIES).float()
        return spec, label

def get_model():
    model = timm.create_model('efficientnet_b0', pretrained=True, in_chans=3)
    model.classifier = nn.Linear(model.classifier.in_features, Config.NUM_SPECIES)
    return model

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for specs, labels in tqdm(dataloader, desc="Training"):
        specs, labels = specs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(specs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

In [ ]:
################################################################################
# PSEUDO LABEL GENERATION AND INFERENCE FUNCTIONS
################################################################################

def run_inference_on_unlabeled(model, unlabeled_files, device):
    """Uses a trained model to generate new pseudo-labels for a list of files."""
    model.eval()
    new_rows = []
    print("Generating new pseudo-labels with the trained model...")
    for path in tqdm(unlabeled_files, desc="Re-labeling"):
        try:
            full_path = os.path.join(Config.UNLABELED_DIR, path)
            waveform, _ = librosa.load(full_path, sr=Config.SAMPLE_RATE, mono=True)
            duration = librosa.get_duration(y=waveform, sr=Config.SAMPLE_RATE)

            for _ in range(Config.PSEUDO_LABEL_SAMPLES_PER_SOUNDSCAPE):
                if len(waveform) < Config.N_SAMPLES: continue
                start_sample = np.random.randint(0, len(waveform) - Config.N_SAMPLES)
                clip = waveform[start_sample : start_sample + Config.N_SAMPLES]

                # Create spectrogram
                spec = torchaudio.transforms.MelSpectrogram(
                    sample_rate=Config.SAMPLE_RATE, n_fft=Config.N_FFT, hop_length=Config.HOP_LENGTH, n_mels=Config.N_MELS
                )(torch.from_numpy(clip).float())
                spec = (spec - spec.mean()) / spec.std()
                spec = spec.unsqueeze(0).repeat(3, 1, 1).unsqueeze(0).to(device) # Add batch dim

                with torch.no_grad():
                    logits = model(spec)
                    probs = torch.sigmoid(logits).squeeze().cpu().numpy()

                primary_label_idx = np.argmax(probs)
                primary_label_str = INDEX_TO_SPECIES[primary_label_idx]
                
                soft_label_str = ','.join(map(str, probs))
                new_rows.append({
                    'filepath': full_path, 'primary_label': primary_label_str,
                    'duration': duration, 'is_pseudo': True,
                    'soft_label': soft_label_str, 'start_time': start_sample / Config.SAMPLE_RATE
                })
        except Exception as e:
            print(f"Error re-labeling {path}: {e}")
            continue
            
    return pd.DataFrame(new_rows)

def run_final_inference_with_ensemble(soundscape_path, ensemble_models, device):
    """
    Performs inference on a single soundscape file using an ensemble of models
    and applies post-processing smoothing.
    """
    print(f"\nRunning inference on {soundscape_path} with ensemble and smoothing...")
    # Load all models into a list
    loaded_models = []
    for model_path in ensemble_models:
        model = get_model()
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.to(device)
        model.eval()
        loaded_models.append(model)
        
    # Load audio and split into non-overlapping chunks
    y, _ = librosa.load(soundscape_path, sr=Config.SAMPLE_RATE, mono=True)
    all_preds = []
    
    for i in range(0, len(y) - Config.N_SAMPLES, Config.N_SAMPLES):
        clip = y[i : i + Config.N_SAMPLES]
        
        # Create spectrogram
        spec_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=Config.SAMPLE_RATE, n_fft=Config.N_FFT, hop_length=Config.HOP_LENGTH, n_mels=Config.N_MELS
        )
        spec = spec_transform(torch.from_numpy(clip).float())
        spec = (spec - spec.mean()) / spec.std()
        spec = spec.unsqueeze(0).repeat(3, 1, 1).unsqueeze(0).to(device)

        with torch.no_grad():
            ensemble_logits = []
            for model in loaded_models:
                logits = model(spec)
                ensemble_logits.append(logits)
            
            # Average the logits from all models in the ensemble
            avg_logits = torch.mean(torch.stack(ensemble_logits), dim=0)
            probs = torch.sigmoid(avg_logits).squeeze().cpu().numpy()
            all_preds.append(probs)

    if not all_preds:
        print("Soundscape too short for inference.")
        return None
        
    # Stack predictions into a [Time, Classes] tensor for smoothing
    preds_tensor = torch.tensor(all_preds, dtype=torch.float32).T.unsqueeze(0) # Shape: [1, Classes, Time]

    # Create and apply the 1D convolution for smoothing
    kernel_size = len(Config.SMOOTHING_KERNEL)
    smoothing_conv = nn.Conv1d(in_channels=Config.NUM_SPECIES, out_channels=Config.NUM_SPECIES,
                               kernel_size=kernel_size, padding='same', bias=False, groups=Config.NUM_SPECIES)
    
    kernel = torch.tensor(Config.SMOOTHING_KERNEL, dtype=torch.float32).view(1, 1, kernel_size)
    smoothing_conv.weight.data = kernel.repeat(Config.NUM_SPECIES, 1, 1)
    
    smoothed_preds = smoothing_conv(preds_tensor).squeeze(0).T.numpy() # Shape: [Time, Classes]

    print("Inference complete. Returning smoothed predictions.")
    return smoothed_preds

In [10]:
################################################################################
# TRAINING AND INFERENCE 
################################################################################


if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using device: MPS (Apple Silicon GPU)")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

# Generate initial pseudo-labels
pseudo_df = generate_initial_pseudo_labels()

# Load original training data
try:
    base_df = pd.read_csv(os.path.join(Config.DATA_DIR, "train_metadata.csv"))
    base_df['filepath'] = Config.AUDIO_DIR + base_df['filename']
    base_df['is_pseudo'] = False
    base_df['soft_label'] = pd.NA
    base_df['start_time'] = 0.0
except FileNotFoundError:
    print("Error: train_metadata.csv not found. Please ensure it's in the data directory.")
    exit()

unlabeled_files = sorted(os.listdir(Config.UNLABELED_DIR))
if Config.ABRIDGED_RUN: unlabeled_files = random.sample(unlabeled_files, 5)


Using device: MPS (Apple Silicon GPU)
Loading cached initial pseudo-labels from initial_pseudo_labels.csv


In [ ]:
# Iterative Training/Pseudo-Labeling
for i in range(Config.NUM_ITERATIONS):
    print(f"\n--- SELF-TRAINING ITERATION {i+1}/{Config.NUM_ITERATIONS} ---")
    train_df = pd.concat([base_df, pseudo_df], ignore_index=True)
    dataset = BirdCLEFDataset(train_df, is_train=True)
    loader = DataLoader(dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    iter_model = get_model().to(device)
    optimizer = optim.Adam(iter_model.parameters(), lr=Config.LEARNING_RATE)
    criterion = nn.BCEWithLogitsLoss()
    for epoch in range(Config.NUM_EPOCHS_PER_ITERATION):
        print(f"Iteration {i+1}, Epoch {epoch+1}/{Config.NUM_EPOCHS_PER_ITERATION}")
        train_loss = train_one_epoch(iter_model, loader, optimizer, criterion, device)
        print(f"Epoch {epoch+1} Train Loss: {train_loss:.4f}")
    # Generate new pseudo-labels for unlabeled files
    pseudo_df = run_inference_on_unlabeled(iter_model, unlabeled_files, device)

In [ ]:
# Ensemble Training
print(f"\n--- FINAL ENSEMBLE TRAINING ---")
final_train_df = pd.concat([base_df, pseudo_df], ignore_index=True)
final_dataset = BirdCLEFDataset(final_train_df, is_train=True)
ensemble_model_paths = []
for i in range(Config.NUM_ENSEMBLE_MODELS):
    print(f"\nTraining Ensemble Model {i+1}/{Config.NUM_ENSEMBLE_MODELS}")
    final_loader = DataLoader(final_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=0)
    model = get_model().to(device)
    optimizer = optim.Adam(model.parameters(), lr=Config.LEARNING_RATE)
    criterion = nn.BCEWithLogitsLoss()
    for epoch in range(Config.NUM_EPOCHS_FINAL):
        print(f"Ensemble Model {i+1}, Epoch {epoch+1}/{Config.NUM_EPOCHS_FINAL}")
        train_loss = train_one_epoch(model, final_loader, optimizer, criterion, device)
        print(f"Epoch {epoch+1} Train Loss: {train_loss:.4f}")
    model_path = os.path.join(Config.CHECKPOINT_DIR, f"{Config.MODEL_NAME}_ensemble_model_{i}.pth")
    torch.save(model.state_dict(), model_path)
    ensemble_model_paths.append(model_path)
    print(f"Saved ensemble model to {model_path}")

In [ ]:
# Inference Demo
if unlabeled_files:
    demo_soundscape = os.path.join(Config.UNLABELED_DIR, unlabeled_files[0])
    smoothed_predictions = run_final_inference_with_ensemble(demo_soundscape, ensemble_model_paths, device)
    if smoothed_predictions is not None:
        print("\nShape of final smoothed predictions [Time, Classes]:", smoothed_predictions.shape)
        top_bird_idx = np.argmax(smoothed_predictions, axis=1)
        print("Top predicted bird for first 5 time steps:", [INDEX_TO_SPECIES[idx] for idx in top_bird_idx[:5]])
